In [40]:
from pathlib import Path
import pandas as pd
import numpy as np
from imblearn.pipeline import make_pipeline
from imblearn.under_sampling import RandomUnderSampler
from sklearn.base import clone
from sklearn.ensemble import (
    RandomForestClassifier,
    HistGradientBoostingClassifier,
    )
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    mutual_info_score,
    balanced_accuracy_score,
    )
from sklearn.model_selection import (
    train_test_split,
    cross_validate,
    )
from sklearn.tree import DecisionTreeClassifier

In [41]:
home = Path.home()
data_path = home / 'Programming/data/fraud-detection/data'
data_df = pd.read_csv(
    data_path / 'transactions_obf.csv', parse_dates=['transactionTime'],
    dtype={'availableCash': np.int64, 'transactionAmount': np.float64}
    )
data_df.sort_values(by='transactionTime', inplace=True)

In [42]:
labels_df = pd.read_csv(
    data_path / 'labels_obf.csv', parse_dates=['reportedTime']
    )
labels_df.sort_values(by='reportedTime', inplace=True)

In [43]:
print(data_df.info())

<class 'pandas.core.frame.DataFrame'>
Index: 118621 entries, 0 to 118619
Data columns (total 10 columns):
 #   Column             Non-Null Count   Dtype              
---  ------             --------------   -----              
 0   transactionTime    118621 non-null  datetime64[ns, UTC]
 1   eventId            118621 non-null  object             
 2   accountNumber      118621 non-null  object             
 3   merchantId         118621 non-null  object             
 4   mcc                118621 non-null  int64              
 5   merchantCountry    118621 non-null  int64              
 6   merchantZip        95616 non-null   object             
 7   posEntryMode       118621 non-null  int64              
 8   transactionAmount  118621 non-null  float64            
 9   availableCash      118621 non-null  int64              
dtypes: datetime64[ns, UTC](1), float64(1), int64(4), object(4)
memory usage: 10.0+ MB
None


In [44]:
# turned some category types represented by integrals such as 'mcc'
# 'merchantCountry' and 'posEntryMode' to strings
category_list = [
    'eventId', 'accountNumber', 'merchantId', 'mcc',
    'merchantCountry', 'merchantZip', 'posEntryMode'
    ]
data_df[category_list] = data_df[category_list].astype('string')

In [45]:
print(data_df.describe())

       transactionAmount  availableCash
count      118621.000000  118621.000000
mean           53.674774    6625.508974
std           183.665315    3410.289486
min            -0.150000     500.000000
25%             8.030000    4500.000000
50%            20.250000    7500.000000
75%            49.000000    8500.000000
max         13348.000000   18500.000000


In [46]:
data_df.head()

,transactionTime,eventId,accountNumber,merchantId,mcc,merchantCountry,merchantZip,posEntryMode,transactionAmount,availableCash
0,2017-01-01 00:00:00+00:00,18688431A1,94f9b4e7,b76d06,5968,826,CR0,1,10.72,7500
3,2017-01-01 00:15:07+00:00,11162049A1,038099dd,7d5803,5499,826,NR1,81,21.00,7500
4,2017-01-01 00:37:09+00:00,17067235A1,3130363b,12ca76,5411,826,M50,81,47.00,10500
2,2017-01-01 00:43:17+00:00,31294145A1,c0ffab1b,94cafc,5735,442,<NA>,81,5.04,9500
1,2017-01-01 00:49:03+00:00,2164986A1,648e19cf,718cc6,5499,826,DE14,81,21.19,4500


In [47]:
print(f'Number of unique categorical entries:\n'
      f'{data_df.select_dtypes('string').nunique().sort_values(ascending=False)}')

Number of unique categorical entries:
eventId            118621
merchantId          33327
merchantZip          3260
accountNumber         766
mcc                   361
merchantCountry        82
posEntryMode           10
dtype: int64


In [48]:
print(f'Number of missing values:\n{data_df.isnull().sum()}')

Number of missing values:
transactionTime          0
eventId                  0
accountNumber            0
merchantId               0
mcc                      0
merchantCountry          0
merchantZip          23005
posEntryMode             0
transactionAmount        0
availableCash            0
dtype: int64


In [49]:
print(f'Percentage of missing values:\n{round(100*data_df.isnull().sum()/data_df.shape[0], 1)}')

Percentage of missing values:
transactionTime       0.0
eventId               0.0
accountNumber         0.0
merchantId            0.0
mcc                   0.0
merchantCountry       0.0
merchantZip          19.4
posEntryMode          0.0
transactionAmount     0.0
availableCash         0.0
dtype: float64


In [50]:
print(f'Sorted list of normalized merchantZip codes:\n{data_df.merchantZip.value_counts(dropna=False, normalize=True)[:10]}')

Sorted list of normalized merchantZip codes:
merchantZip
<NA>    0.193937
0       0.122019
E12     0.009644
SL4     0.005496
LS11    0.005151
CO10    0.004637
HA8     0.004578
AL10    0.004578
NW4      0.00446
IV30    0.004426
Name: proportion, dtype: Float64


In [51]:
print(f'''Percentage of NAs and 0s in 'merchantZip': {round(100*((data_df.merchantZip == '0').sum() + data_df.merchantZip.isna().sum())/data_df.shape[0], 2)}%.''')

Percentage of NAs and 0s in 'merchantZip': 31.6%.


In [52]:
data_df['merchantZip'] = data_df['merchantZip'].replace({np.nan: 'Unknown', '0': 'Unknown'})

In [53]:
print(f'Sorted list of normalized merchantZip codes:\n{data_df.merchantZip.value_counts(dropna=False, normalize=True)[:10]}')

Sorted list of normalized merchantZip codes:
merchantZip
Unknown    0.315956
E12        0.009644
SL4        0.005496
LS11       0.005151
CO10       0.004637
HA8        0.004578
AL10       0.004578
NW4         0.00446
IV30       0.004426
TD1        0.003861
Name: proportion, dtype: Float64


In [54]:
print(f'Sorted list of normalized posEntryMode codes:\n{data_df.posEntryMode.value_counts(dropna=False, normalize=True)[:10]}')

Sorted list of normalized posEntryMode codes:
posEntryMode
5     0.592037
81    0.302071
1      0.08909
90    0.010133
7      0.00537
80    0.000776
79    0.000261
2     0.000135
0     0.000093
91    0.000034
Name: proportion, dtype: Float64


In [55]:
print(labels_df.info())

<class 'pandas.core.frame.DataFrame'>
Index: 875 entries, 0 to 874
Data columns (total 2 columns):
 #   Column        Non-Null Count  Dtype              
---  ------        --------------  -----              
 0   reportedTime  875 non-null    datetime64[ns, UTC]
 1   eventId       875 non-null    object             
dtypes: datetime64[ns, UTC](1), object(1)
memory usage: 20.5+ KB
None


In [56]:
labels_df['eventId'] = labels_df['eventId'].astype('string')

In [57]:
print(f'Number of unique entries in labels dataset:\n{labels_df.nunique()}')

Number of unique entries in labels dataset:
reportedTime    145
eventId         875
dtype: int64


In [58]:
data_df['fraudCase'] = data_df.eventId.isin(labels_df.eventId).astype(bool)

In [59]:
neg, pos = np.bincount(data_df['fraudCase'])
total = neg + pos
print(
    f'Total number of transactions: {total}\n'
    f'Number of non-fraudulent transactions: {neg} ({100 * neg / total:.2f}% of total)\n'
    f'Number of fraudulent transactions: {pos} ({100 * pos / total:.2f}% of total)\n'
    )


Total number of transactions: 118621
Number of non-fraudulent transactions: 117746 (99.26% of total)
Number of fraudulent transactions: 875 (0.74% of total)



In [60]:
# information on the data set
total_num_accounts = data_df['accountNumber'].nunique()
fraud_df = data_df.loc[data_df.fraudCase == 1]
grouped = fraud_df.groupby('accountNumber')['transactionAmount']
fraud_amount_per_account_df = grouped.sum()
num_accounts_with_fraud = len(fraud_amount_per_account_df)
num_accounts_frauds_less_than_1000 = np.sum(fraud_amount_per_account_df < 1000)

In [61]:
results_dict = {
    'total_num_accounts': total_num_accounts,
    'num_accounts_with_fraud': num_accounts_with_fraud,
    'percent_fraud_per_transaction':np.round(
        100*((data_df['fraudCase'] == 1).sum()/data_df.shape[0]), 2
        ),
    'percent_accounts_with_fraud': np.round(
        100*num_accounts_with_fraud/total_num_accounts, 2
        ),
    'percent_frauds_less_than_1000_GBP': np.round(
        100*(num_accounts_frauds_less_than_1000/num_accounts_with_fraud), 2),
}

In [62]:
print(f'''Total number of accounts: {
    results_dict['total_num_accounts']}'''
    )
print(f'''Number of accounts with fraud: {
    results_dict['num_accounts_with_fraud']}'''
    )
print(f'''Percentage of fraud per transaction: {
    results_dict['percent_fraud_per_transaction']}%'''
    )
print(f'''Percentage of accounts with fraud: {
    results_dict['percent_accounts_with_fraud']}%'''
    )
print(f'''Percentage of accounts with less than £1000 of fraud: {
    results_dict['percent_frauds_less_than_1000_GBP']}%'''
    )

Total number of accounts: 766
Number of accounts with fraud: 167
Percentage of fraud per transaction: 0.74%
Percentage of accounts with fraud: 21.8%
Percentage of accounts with less than £1000 of fraud: 83.23%


In [63]:
def mutual_info(x):
    return mutual_info_score(x, data_df.fraudCase)

In [64]:
categorical_list = list(data_df.select_dtypes('string'))
numerical_list = list(data_df.select_dtypes('number'))

In [65]:
print(f'Mutual information score:\n{data_df[categorical_list].apply(mutual_info).sort_values(ascending=False)}')

Mutual information score:
eventId            0.043564
merchantId         0.031126
accountNumber      0.020102
merchantZip        0.008898
mcc                0.008623
merchantCountry    0.006532
posEntryMode       0.006042
dtype: float64


In [66]:
corr_df = data_df[numerical_list].corr()
print(f'Correlation matrix of numerical features:\n{corr_df}')

Correlation matrix of numerical features:
                   transactionAmount  availableCash
transactionAmount           1.000000       0.063817
availableCash               0.063817       1.000000


In [67]:
data_df.drop(['eventId', 'transactionTime', 'merchantId'], axis=1, inplace=True)
updated_categorical_list = list(data_df.select_dtypes('string'))

In [68]:
selected_list = updated_categorical_list + numerical_list
dicts = data_df[selected_list].to_dict(orient='records')
dv = DictVectorizer(sparse=False)
dicts_arr = dv.fit_transform(dicts)
print(f'Shape of processed dataframe: {dicts_arr.shape}')

Shape of processed dataframe: (118621, 4481)


In [69]:
feature_names_list = dv.get_feature_names_out(selected_list)

In [70]:
processed_df = pd.DataFrame(dicts_arr, columns=feature_names_list)

In [71]:
y = data_df['fraudCase']
X = processed_df

In [72]:
# training set: first 10 months
len_monthly_data_set = int(X.shape[0]/12)
X_train_full = X[:len(X)-2*len_monthly_data_set]
y_train_full = y[:len(y)-2*len_monthly_data_set]

# test set: last two months 
X_test = X[len(X)-2*len_monthly_data_set:]
y_test = y[len(y)-2*len_monthly_data_set:]

# training set: first 8 months
X_train = X_train_full[:len(X_train_full)-2*len_monthly_data_set]
y_train = y_train_full[:len(y_train_full)-2*len_monthly_data_set]

# validation set: 9th and 10th month
X_val = X_train_full[len(X_train_full)-2*len_monthly_data_set:]
y_val = y_train_full[len(y_train_full)-2*len_monthly_data_set:]

In [73]:
# first test month
X_test_1 = X_test[:len_monthly_data_set]
y_test_1 = y_test[:len_monthly_data_set]
# second test month
X_test_2 = X_test[len_monthly_data_set:]
y_test_2 = y_test[len_monthly_data_set:]

In [74]:
print(f'Length of full training set (first 10 months): {len(X_train_full)}')
print(f'Length of full testing set (last 2 months): {len(X_test)}')
print(f'Length of training set (first 8 months): {len(X_train)}')
print(f'Length of 1st validation set (9th & 10th months): {len(X_val)}')
print(f'{len(X_train)+len(X_val)+len(X_test) == len(X)}')

Length of full training set (first 10 months): 98851
Length of full testing set (last 2 months): 19770
Length of training set (first 8 months): 79081
Length of 1st validation set (9th & 10th months): 19770
True


In [75]:
model_dict = {
    'logistic regression': LogisticRegression(max_iter=4000, random_state=0,),
    'decision tree': DecisionTreeClassifier(max_depth=6, random_state=0),
    'random forest': RandomForestClassifier(
        max_depth=8, n_estimators=200, n_jobs=-1,
        random_state=0, bootstrap=True,
        ),
    'histogram gradient boosting': HistGradientBoostingClassifier(
        max_depth=8, max_iter=400, random_state=0, early_stopping=True,
        ),
    }

In [76]:
list_of_models = list(model_dict.keys())
mod_list = str(list_of_models)[1:-1].split(', ')
new_list = ', '.join(mod_list[:-1]) + ' and ' + ' '.join(mod_list[-1:])
print(f'Modeling data on {new_list}:\n')

Modeling data on 'logistic regression', 'decision tree', 'random forest' and 'histogram gradient boosting':



In [77]:
class DataSetEvaluation():
    def __init__(self, X_train, X_val, y_train, y_val):
        self.X_train = X_train
        self.X_val = X_val
        self.y_train = y_train
        self.y_val = y_val

    def cv_evaluate(self):
        for name, model in model_dict.items():
            pipeline = make_pipeline(
                RandomUnderSampler(random_state=0),
                model
                )
            cv_res = cross_validate(
                pipeline, self.X_train, self.y_train,
                scoring="balanced_accuracy",
                return_train_score=True, n_jobs=-1,
                return_estimator=True, error_score='raise'
                )
            print(f"Balanced accuracy mean ± std. dev. for {name}: "
                  f"{np.round(cv_res['test_score'].mean(), 3)} ± "
                  f"{np.round(cv_res['test_score'].std(), 3)}")
            scores = []
            for cv_model in cv_res["estimator"]:
                scores.append(
                    balanced_accuracy_score(
                        self.y_val, cv_model.predict(self.X_val)
                        )
                    )
            print(f"Balanced accuracy mean ± std. dev. "
                  f"for {name} on validation dataset: "
                  f"{np.round(np.mean(scores), 3)} ± "
                  f"{np.round(np.std(scores), 3)}"
                  f"\n")

In [78]:
validation_eval = DataSetEvaluation(X_train, X_val, y_train, y_val)
validation_eval.cv_evaluate()

Balanced accuracy mean ± std. dev. for logistic regression: 0.818 ± 0.099
Balanced accuracy mean ± std. dev. for logistic regression on validation dataset: 0.815 ± 0.026

Balanced accuracy mean ± std. dev. for decision tree: 0.796 ± 0.073
Balanced accuracy mean ± std. dev. for decision tree on validation dataset: 0.78 ± 0.027

Balanced accuracy mean ± std. dev. for random forest: 0.814 ± 0.05
Balanced accuracy mean ± std. dev. for random forest on validation dataset: 0.819 ± 0.009

Balanced accuracy mean ± std. dev. for histogram gradient boosting: 0.835 ± 0.08
Balanced accuracy mean ± std. dev. for histogram gradient boosting on validation dataset: 0.812 ± 0.02



In [79]:
pipeline = make_pipeline(
    RandomUnderSampler(random_state=0),
    RandomForestClassifier(
        max_depth=8, n_estimators=200, n_jobs=-1,
        random_state=0, bootstrap=True,
        )
    )
pipeline.fit(X_train, y_train)

,steps,"[('randomundersampler', ...), ('randomforestclassifier', ...)]"
,transform_input,None
,memory,None
,verbose,False
,sampling_strategy,'auto'
,random_state,0
,replacement,False
,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",200
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",8
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2


In [82]:
class ModelOverRandomDetection():
    '''
    Checks performance of fraud detection in model over random-transaction
    selection. The number of bootstrapped cases and the number of transaction
    checks can be changed for each instance of the class in the 'eval' method.
    '''

    def __init__(self, X_test, y_test, pipeline):
        self.X_test = X_test
        self.y_test = y_test
        self.pipeline = pipeline

    def eval(self, num_bootstrapped_cases=30, num_transaction_checks=400):
        self.num_transaction_checks = num_transaction_checks
        self.num_bootstrapped_cases = num_bootstrapped_cases
        self.percent_frauds_control = []

        # test set
        fraud_prob = self.pipeline.predict_proba(self.X_test)[:, 1]
        ps = fraud_prob.argsort() 
        index_selected_trans = ps[-self.num_transaction_checks:]
        y_sel = self.y_test.iloc[index_selected_trans]
        self.percent_frauds = 100*np.sum(y_sel)/np.sum(self.y_test)

        # control on random selection
        for n_tests in range(self.num_bootstrapped_cases):

              rng = np.random.RandomState(seed=n_tests)
              shuffled_index = rng.permutation(np.arange(0, len(self.y_test)))
              selected_indexes = shuffled_index[:num_transaction_checks]
              sens = 100*sum(self.y_test.iloc[selected_indexes])/sum(self.y_test)
              self.percent_frauds_control.append(sens)

        print(f'Average detection percentage across '
              f'{self.num_bootstrapped_cases} bootstrapped datasets:')

        print(f'Model fraud detection accuracy on test set: '
              f'{np.round(self.percent_frauds, 2)}%')

        percent_frauds_control_mean = np.mean(self.percent_frauds_control)
        print(f'Random fraud detection accuracy on test set: '
              f'{np.round(percent_frauds_control_mean, 2)}%\n')

        print(f'Improvement of model accuracy over random fraud detection on test set: '
              f'{np.round(self.percent_frauds/percent_frauds_control_mean, 1)}x')



In [83]:
test_1_eval = ModelOverRandomDetection(X_test_1, y_test_1, pipeline)
test_1_eval.eval()

Average detection percentage across 30 bootstrapped datasets:
Model fraud detection accuracy on test set: 27.78%
Random fraud detection accuracy on test set: 5.19%

Improvement of model accuracy over random fraud detection on test set: 5.4x


In [84]:
test_2_eval = ModelOverRandomDetection(X_test_2, y_test_2, pipeline)
test_2_eval.eval()

Average detection percentage across 30 bootstrapped datasets:
Model fraud detection accuracy on test set: 20.0%
Random fraud detection accuracy on test set: 3.25%

Improvement of model accuracy over random fraud detection on test set: 6.2x
